# Ingestion Pre/Post Operations

The ingestion pipeline supports pluggable **pre-operations** and **post-operations**
that run before and after the main data processing. These are configured per-request
via the `pre_operations` and `post_operations` fields in the ingestion body.

## Architecture

```
Ingestion Task
  │
  ├─ 1. Resolve asset (from URI or asset_id)
  ├─ 2. Run PRE-OPERATIONS (sequential, can modify asset)
  │     ├─ e.g. download from URL → GCS bucket
  │     └─ e.g. detect encoding → override task_request.encoding
  ├─ 3. Main ingestion (read file, map columns, upsert batches)
  └─ 4. Run POST-OPERATIONS (parallel, receive status)
        ├─ e.g. send notification
        └─ e.g. trigger downstream pipeline
```

### Key difference

- **Pre-operations** run sequentially and can **modify** the asset (e.g., download to local path)
- **Post-operations** run in parallel and receive the final `status` (`COMPLETED` or `FAILED`)

## `IngestionOperationInterface`

All operations implement this abstract base class:

```python
class IngestionOperationInterface(Generic[T_CONFIG], ABC):
    def __init__(self, engine, task_id, task_request,
                 catalog_config=None, ingestion_config=None,
                 config: Optional[T_CONFIG] = None):
        ...

    @abstractmethod
    async def pre_op(self, catalog, collection, asset) -> asset:
        """Runs before ingestion. Return modified or original asset."""

    @abstractmethod
    async def post_op(self, catalog, collection, asset, status, error_message=None):
        """Runs after ingestion with outcome status."""
```

Operations are registered via the `@ingestion_operation` decorator and discovered
by class name in the request body.

In [ ]:
# === Parameters ===
import os
BASE = os.environ.get("DYNASTORE_BASE_URL") or "http://localhost:8080"
CATALOG_ID = "demo-operations"
COLLECTION_ID = "weather-data"

In [ ]:
import httpx
import json

client = httpx.AsyncClient(base_url=BASE, timeout=60)

r = await client.post("/stac/catalogs", json={"id": CATALOG_ID, "title": "Operations Demo"})
print("Catalog:", r.status_code)

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID,
    "title": "Weather Observations",
    "license": "CC-BY-4.0",
})
print("Collection:", r.status_code)

---
## 1. Pre-operations

Pre-operations execute before the main ingestion loop. Each operation receives
the current asset and can return a modified version (e.g., with a new local `uri`
after downloading from a remote URL).

### Configuration in the ingestion request

The `pre_operations` field is a dictionary where:
- **Keys** are the registered operation class names
- **Values** are the operation-specific configuration (validated as a Pydantic model)

### Built-in pre-operations (GCP module)

| Operation | Purpose | Config |
|-----------|---------|--------|
| `ContentTypeInspector` | Detects file encoding via byte sampling | `{"sample_size": 50000}` |
| `AssetDownloader` | Downloads remote URL to GCS bucket | `{"target_bucket": "..."}` |

### Example: encoding detection + asset download

```json
{
    "inputs": {
        "catalog_id": "my-catalog",
        "collection_id": "my-collection",
        "ingestion_request": {
            "asset": {"uri": "https://example.com/data/stations.zip"},
            "column_mapping": {
                "external_id": "station_id",
                "csv_lat_column": "lat",
                "csv_lon_column": "lon"
            },
            "pre_operations": {
                "AssetDownloader": {
                    "target_bucket": "my-catalog-assets"
                },
                "ContentTypeInspector": {
                    "sample_size": 50000
                }
            }
        }
    }
}
```

The operations run in declaration order:
1. `AssetDownloader` downloads the ZIP from the URL to a GCS bucket, updates the asset URI
2. `ContentTypeInspector` samples bytes from the downloaded file to detect encoding

After pre-operations, the main ingestion loop uses the updated asset URI and detected encoding.

In [ ]:
# Example ingestion body with pre-operations
# (operations must be registered in your deployment's SCOPE)

body_with_pre_ops = {
    "inputs": {
        "catalog_id": CATALOG_ID,
        "collection_id": COLLECTION_ID,
        "ingestion_request": {
            "asset": {"uri": "https://data.example.org/weather/2024.csv"},
            "encoding": "utf-8",
            "column_mapping": {
                "external_id": "observation_id",
                "csv_lat_column": "lat",
                "csv_lon_column": "lon",
                "attributes_source_type": "all"
            },
            "pre_operations": {
                "ContentTypeInspector": {
                    "sample_size": 50000
                }
            }
        }
    }
}

print("Ingestion with pre-operations:")
print(json.dumps(body_with_pre_ops, indent=2))

---
## 2. Post-operations

Post-operations execute **after** the main ingestion loop completes (or fails).
They run in **parallel** via `asyncio.gather` and receive the final status.

### Use cases

- Send a webhook notification on completion/failure
- Trigger a downstream ETL pipeline
- Clean up temporary files
- Update an external metadata catalog

### Configuration

Same pattern as pre-operations — a dictionary of `{ClassName: config}`:

```json
{
    "ingestion_request": {
        "post_operations": {
            "WebhookNotifier": {
                "url": "https://hooks.example.com/ingestion-done",
                "include_stats": true
            }
        }
    }
}
```

### Status values received by post-operations

| Status | Meaning |
|--------|---------|
| `"COMPLETED"` | Ingestion finished successfully |
| `"FAILED"` | Ingestion failed; `error_message` contains the reason |

In [ ]:
# Example combining pre and post operations

full_ops_body = {
    "inputs": {
        "catalog_id": CATALOG_ID,
        "collection_id": COLLECTION_ID,
        "ingestion_request": {
            "asset": {"uri": "/data/weather_stations_2024.csv"},
            "column_mapping": {
                "external_id": "obs_id",
                "csv_lat_column": "latitude",
                "csv_lon_column": "longitude",
                "attributes_source_type": "all"
            },
            "pre_operations": {
                "ContentTypeInspector": {"sample_size": 50000}
            },
            "post_operations": {
                "WebhookNotifier": {
                    "url": "https://hooks.example.com/done",
                    "include_stats": True
                }
            }
        }
    }
}

print("Full ingestion body with pre + post operations:")
print(json.dumps(full_ops_body, indent=2))

---
## 3. Writing a custom operation

To create your own operation, implement `IngestionOperationInterface` with a
typed Pydantic config model and register it with the `@ingestion_operation` decorator.

```python
from pydantic import BaseModel
from dynastore.tasks.ingestion.operations import (
    IngestionOperationInterface,
    ingestion_operation,
)


class SridOverrideConfig(BaseModel):
    """Config model — validated from the request body automatically."""
    target_srid: int = 4326
    reproject: bool = False


@ingestion_operation
class SridOverride(IngestionOperationInterface[SridOverrideConfig]):
    """Override the source SRID before ingestion starts."""

    async def pre_op(self, catalog, collection, asset):
        if self.config:
            self.task_request.source_srid = self.config.target_srid
            # Optionally access self.catalog_config, self.ingestion_config
        return asset  # return the (possibly modified) asset

    async def post_op(self, catalog, collection, asset, status, error_message=None):
        if status == "FAILED":
            logger.warning(f"Ingestion failed for asset {asset.asset_id}: {error_message}")
```

### Usage in the request body

```json
{
    "pre_operations": {
        "SridOverride": {
            "target_srid": 32632,
            "reproject": true
        }
    }
}
```

### Context available in operations

| Attribute | Type | Description |
|-----------|------|-------------|
| `self.engine` | `Engine` | Database engine for any DB operations |
| `self.task_id` | `str` | Running task ID |
| `self.task_request` | `TaskIngestionRequest` | Full ingestion request (mutable for pre-ops) |
| `self.catalog_config` | `DriverRecordsPostgresqlConfig` | Physical storage config |
| `self.ingestion_config` | `IngestionPluginConfig` | Mutable ingestion logic config |
| `self.config` | `T_CONFIG` | Your operation's validated Pydantic config |

In [ ]:
# Cleanup
r = await client.delete(f"/stac/catalogs/{CATALOG_ID}?force=true")
print("Cleanup:", r.status_code)
await client.aclose()